In [ ]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib as mpl
from astropy.coordinates import SkyCoord,EarthLocation, AltAz, ICRS, Galactic, FK4, FK5
import matplotlib.gridspec as gridspec
import seaborn as sns
from astropy.wcs import WCS
from scipy.stats import rayleigh

In [ ]:
dir_in = '/srv/data/dva/dragons_hoard/Data_cubes_Sept_2025_BB/convolved_masked_cubes/350to1000/conv_plat_car/'

dir_in = '/srv/data/dragons/dragons-canfar-Nov2025/'

hdu_dirty = fits.open(dir_in + 'faraday_synthesis/dragons_FDF_tot_dirty_Kgal.car.fits')
hdu_clean = fits.open(dir_in + 'faraday_synthesis/dragons_FDF_clean_tot_Kgal.car.fits')
hdu_cc    = fits.open(dir_in + 'faraday_synthesis/dragons_FDF_CC_tot_Kgal.car.fits')
#hdu_thresh = fits.open(dir_in + 'rmsynth_coarse/three_sigma.fits')
#hdu_niter = fits.open(dir_in + 'rmsynth/clean_5sigma/CLEAN_nIter.fits')
hdu_rmsf = fits.open(dir_in + 'faraday_synthesis/dragons_RMSF_tot_Kgal.car.fits')
hdu_rmsf_w = fits.open(dir_in + 'faraday_synthesis/dragons_FDF_RMSF_FWHM_gal.car.fits')

hdu_Q = fits.open(dir_in + 'frequency_cubes/dragons_freqK_gal_conv_q.car.fits')
hdu_U = fits.open(dir_in + 'frequency_cubes/dragons_freqK_gal_conv_u.car.fits')
hdu_I = fits.open(dir_in + 'frequency_cubes/dragons_freqK_gal_conv_i.car.fits')
hdu_dQU = fits.open(dir_in + 'frequency_cubes/dragons_freq_dQU_gal.car.fits')

hdu_sigma = fits.open(dir_in + 'faraday_synthesis/dragons_FDF_sigma_Kgal.car.fits')

hdu_M0 = fits.open(dir_in + 'faraday_synthesis/dragons_M0_gal.car.fits')
hdu_M1 = fits.open(dir_in + 'faraday_synthesis/dragons_M1_gal.car.fits')
hdu_m2 = fits.open(dir_in + 'faraday_synthesis/dragons_m2_gal.car.fits')


In [ ]:
FDF_dirty = hdu_dirty[0].data
FDF_clean = hdu_clean[0].data
#threshold = hdu_thresh[0].data
threshold = hdu_sigma[0].data[0]

Qcube   = hdu_Q[0].data
Ucube   = hdu_U[0].data
Icube   = hdu_I[0].data
dQUcube = hdu_dQU[0].data

M0 = hdu_M0[0].data
M1 = hdu_M1[0].data
m2 = hdu_m2[0].data

sigma = hdu_sigma[0].data
cc = hdu_cc[0].data
#N_iter = hdu_niter[0].data[0]
rmsf = hdu_rmsf[0].data
rmsf_w = hdu_rmsf_w[0].data[0]

wcs_FD   = WCS(hdu_dirty[0].header)
wcs_freq = WCS(hdu_Q[0].header)
wcs_rmsf = WCS(hdu_rmsf[0].header)

print(wcs_FD)
print('')
print(wcs_freq)

In [ ]:
print(FDF_dirty.shape)
print(FDF_clean.shape)
print(threshold.shape)
print(rmsf_w.shape)
print(M0.shape)
print(M1.shape)
print(m2.shape)

In [ ]:
FDvals = wcs_FD.all_pix2world(0,0,range(FDF_dirty.shape[0]),0)[2]
lat_FD = wcs_FD.all_pix2world(0,range(FDF_dirty.shape[1]),0,0)[1]
lon_FD = wcs_FD.all_pix2world(range(FDF_dirty.shape[2]),0,0,0)[0]

freqs    = wcs_freq.all_pix2world(0,0,range(Qcube.shape[0]),0)[2]
lat_freq = wcs_freq.all_pix2world(0,range(Qcube.shape[1]),0,0)[1]
lon_freq = wcs_freq.all_pix2world(range(Qcube.shape[2]),0,0,0)[0]

FDvals_rmsf = wcs_rmsf.all_pix2world(0,0,range(rmsf.shape[0]),0)[2]

In [ ]:
dQUcube[dQUcube < 0] = np.nan

In [ ]:
def make_spectra_plots(lidx,bidx):

    sb_cmap = sns.color_palette("deep") 

    fs     = 26

    #print(N_iter[bidx,lidx])
    print(f'noise: {threshold[bidx,lidx]}')
    print(f'RMSF width: {rmsf_w[bidx,lidx]} rad/m^2')
    print(f'M0: {M0[bidx,lidx]} K/RMSF rad/m^2')
    print(f'M1: {M1[bidx,lidx]} rad/m^2')
    print(f'm2: {m2[bidx,lidx]} rad/m^2')
    
    fig = plt.figure(figsize=(14,17))
    
    plt.subplots_adjust(left=0.11, 
                        bottom=0.08, 
                        right=0.97, 
                        top=0.98,
                        hspace=0.22)
    
    gs = gridspec.GridSpec(ncols=1, nrows=3, figure=fig)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    ax3 = fig.add_subplot(gs[2])

    ax1.errorbar(freqs/1e6, Qcube[:,bidx,lidx], yerr=dQUcube[:,bidx,lidx], fmt='o',label='Stokes $Q$', 
                 elinewidth=2, markerfacecolor = 'k', color='k',alpha=0.8)
    ax1.errorbar(freqs/1e6, Ucube[:,bidx,lidx], yerr=dQUcube[:,bidx,lidx], fmt='o',label='Stokes $U$', 
                 elinewidth=2, markerfacecolor = 'darkgrey', color='darkgrey',alpha=0.8)
    ax1.set_xlim(350,1030)
    ax1.set_ylim(-2.5,2.5)
    ax1.set_xlabel(r'Frequency (MHz)',fontsize=fs)
    ax1.set_ylabel(r'Stokes $Q$, $U$ (K)',fontsize=fs)

    ax2.plot(FDvals, FDF_dirty[:,bidx,lidx],linewidth=2.5, color='darkgrey', label='dirty')
    ax2.plot(FDvals, FDF_clean[:,bidx,lidx],linewidth=2.5, color='k', label='cleaned')
    ax2.plot(FDvals, cc[:,bidx,lidx],linewidth=2.5, color=sb_cmap[0], label='clean components')
    #ax2.axhline(y = 3*threshold[bidx,lidx],linewidth=2.5, color='green', linestyle='dashed', label='clean threshold x3')
    ax2.axhline(y = 5*threshold[bidx,lidx],linewidth=2.5, color=sb_cmap[2], linestyle='dashed', label='clean threshold')
    ax2.axhline(y = 6*threshold[bidx,lidx],linewidth=2.5, color=sb_cmap[4], linestyle='dashed', label='moments threshold')
    ax2.set_xlim(-50,50)
    ax2.set_ylim(0,)
    ax2.set_xlabel(r'Faraday Depth (rad m$^{-2}$)',fontsize=fs)
    ax2.set_ylabel(r'PI (K RMSF$^{-1}$)',fontsize=fs)

    ax3.plot(FDvals_rmsf, rmsf[:,bidx,lidx],linewidth=2.5, color='k', label='FWHM='+str(np.round(rmsf_w[bidx,lidx],2)))
    ax3.set_xlim(-5,5)
    ax3.set_ylim(0,)
    ax3.set_xlabel(r'Faraday Depth (rad m$^{-2}$)',fontsize=fs)
    ax3.set_ylabel(r'RMSF',fontsize=fs)
    ax3.grid(True)
    ax3.axhline(y=0.5)
    
    for ax in [ax1, ax2, ax3]:
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
        
        ax.tick_params(axis='both', labelsize=fs, left=True, right=True, 
                           top=True, which='both', width=2, length=6)
        #if ax in [ax1, ax2]:
        ax.legend(fontsize=fs)

    return

In [ ]:
lpick = 170
bpick = -20

lidx = abs(lpick - lon_FD).argmin()
bidx = abs(bpick - lat_FD).argmin()

print(lidx, lon_FD[lidx], lon_freq[lidx])
print(bidx, lat_FD[bidx], lat_freq[bidx])

make_spectra_plots(lidx,bidx)